In [1]:
# !pip install optuna

In [2]:
# !pip install cmaes

In [3]:
import optuna
import numpy as np
import os
from optuna.samplers import CmaEsSampler

c:\Users\sanja\miniconda3\envs\bo\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def death_penalty(constraints):
    if np.any(constraints > 0):  
        return float('inf')  
    return 0

In [5]:
def logarithmic_barrier(constraints, mu=1000.0):
    # Ensure all constraints are strictly positive (feasible)
    feasible_constraints = constraints[constraints > 0]
    
    # Apply the logarithmic barrier only to feasible constraints
    if len(feasible_constraints) > 0:
        penalty = -mu * np.sum(np.log(feasible_constraints))
    else:
        # If all constraints are violated, return a large penalty
        penalty = np.inf
    return penalty

In [6]:
def exponential_penalty(constraints, mu = 1000):
    return mu * np.sum(np.exp(np.maximum(0, np.array(constraints))) - 1)

In [7]:
def quadratic_penalty(constraints, mu = 1000):
    return mu * np.sum(np.maximum(0, constraints)**2)

In [8]:
def scaobra_penalty(constraints, mu = 1000):
    return mu * np.sum(constraints > 0)

In [9]:
def penalty(constraints):
    return quadratic_penalty(constraints)

In [10]:
def g1_cons(x):
    g1 = 2*x[0]+2*x[1]+x[9]+x[10] - 10
    g2 = 2*x[0]+2*x[2]+x[9]+x[11] - 10
    g3 = 2*x[1]+2*x[2]+x[10]+x[11] - 10
    
    g4 = -8*x[0]+x[9]
    g5 = -8*x[1]+x[10]
    g6 = -8*x[2]+x[11]
    
    g7 = -2*x[3]-x[4]+x[9]
    g8 = -2*x[5]-x[6]+x[10]
    g9 = -2*x[7]-x[8]+x[11]

    return np.array([g1, g2, g3, g4, g5, g6, g7, g8, g9])

In [11]:
#[8,9,10]
def g1(trial):
    x1 = trial.suggest_float("x1", 0, 1)
    x2 = trial.suggest_float("x2", 0, 1)
    x3 = trial.suggest_float("x3", 0, 1)
    x4 = trial.suggest_float("x4", 0, 1)
    x5 = trial.suggest_float("x5", 0, 1)
    x6 = trial.suggest_float("x6", 0, 1)
    x7 = trial.suggest_float("x7", 0, 1)
    x8 = trial.suggest_float("x8", 0, 1)
    x9 = trial.suggest_float("x9", 0, 1)
    x10 = trial.suggest_int("x10", 0, 100)
    x11 = trial.suggest_int("x11", 0, 100)
    x12 = trial.suggest_int("x12", 0, 100)
    x13 = trial.suggest_float("x13", 0, 1)
    x = np.array([x1, x2, x3, x4, x5, x6, x7, x8, x9, x10, x11, x12, x13])
    cons = g1_cons(x)
    penalty_score = penalty(cons)
    obj = np.sum(5*x[:4])-(5*np.sum(x[:4]**2))-(np.sum(x[4:13]))
    return obj + penalty_score

def g1_og(x):
    x = np.array(x)
    return np.sum(5*x[:4])-(5*np.sum(x[:4]**2))-(np.sum(x[4:13])) 

In [12]:
def g2_cons(x):
    g1 = 0.75 - np.prod(x)
    g2 = np.sum(x) - 7.5 * len(x)

    return np.array([g1, g2])

In [13]:
#[1,4]
def g2(trial):
    x1 = trial.suggest_float("x1", 0, 10)
    x2 = trial.suggest_int("x2", 0, 10)
    x3 = trial.suggest_float("x3", 0, 10)
    x4 = trial.suggest_float("x4", 0, 10)
    x5 = trial.suggest_int("x5", 0, 10)
    x6 = trial.suggest_float("x6", 0, 10)
    x = np.array([x1, x2, x3, x4, x5, x6])
    cons = g2_cons(x)
    penalty_score = penalty(cons)
    obj = np.abs((np.sum(np.cos(x)**4) - 2 * np.prod(np.cos(x)**2))/  np.sqrt(np.sum(np.arange(1,len(x)+1) * x**2)))
    return -obj + penalty_score

def g2_og(x):
    x = np.array(x)
    obj = np.abs((np.sum(np.cos(x)**4) - 2 * np.prod(np.cos(x)**2))/  np.sqrt(np.sum(np.arange(1,len(x)+1) * x**2)))
    return -obj

In [14]:
def g3_cons(x):
    return np.array(np.square(x)) - 1

In [15]:
def g3(trial):
    x1 = trial.suggest_int("x1", 0, 1)
    x2 = trial.suggest_float("x2", 0, 1)
    x3 = trial.suggest_float("x3", 0, 1)
    x4 = trial.suggest_int("x4", 0, 1)
    x5 = trial.suggest_float("x5", 0, 1)
    x = np.array([x1, x2, x3, x4, x5])
    n = 5
    cons = g3_cons(x)
    penalty_score = penalty(cons)
    obj = (np.sqrt(n) ** n) * np.prod(x)
    return -obj + penalty_score

def g3_og(x):
    x = np.array(x)
    n = 5
    obj = (np.sqrt(n) ** n) * np.prod(x)
    return -obj

In [16]:
def g4_cons(x):
    g1 = 85.334407 + 0.0056858 * x[1] * x[4] + 0.0006262 * x[0] * x[3] - 0.0022053 * x[2] * x[4] - 92
    g2 = -85.334407 - 0.0056858 * x[1] * x[4] - 0.0006262 * x[0] * x[3] + 0.0022053 * x[2] * x[4]
    g3 = 80.51249 + 0.0071317 * x[1] * x[4] + 0.0029955 * x[0] * x[1] + 0.0021813 * x[2]**2 - 110
    g4 = -80.51249 - 0.0071317 * x[1] * x[4] - 0.0029955 * x[0] * x[1] - 0.0021813 * x[2]**2 + 90
    g5 = 9.300961 + 0.0047026 * x[2] * x[4] + 0.0012547 * x[0] * x[2] + 0.0019085 * x[2] * x[3] - 25
    g6 = -9.300961 - 0.0047026 * x[2] * x[4] - 0.0012547 * x[0] * x[2] - 0.0019085 * x[2] * x[3] + 20
    return np.array([g1, g2, g3, g4, g5, g6])

In [17]:
def g4(trial):
    x1 = trial.suggest_int("x1", 78, 102)
    x2 = trial.suggest_float("x2", 33, 45)
    x3 = trial.suggest_int("x3", 27, 45)
    x4 = trial.suggest_float("x4", 27, 45)
    x5 = trial.suggest_float("x5", 27, 45)
    x = np.array([x1, x2, x3, x4, x5])
    cons = g4_cons(x)
    penalty_score = penalty(cons)
    obj = (5.3578547 * x[2]**2 + 0.8356891 * x[0] * x[4] + 37.293239 * x[0] - 40792.141)
    return obj + penalty_score


def g4_og(x):
    x = np.array(x) 
    obj = (5.3578547 * x[2]**2 + 0.8356891 * x[0] * x[4] + 37.293239 * x[0] - 40792.141)
    return obj

In [18]:
def g5_cons(x):
    g1 = -x[3] + x[2] - 0.55
    g2 = -x[2] + x[3] - 0.55

    # Equality constraints
    h3 = 1000*np.sin(-x[2] - 0.25) + 1000*np.sin(-x[3] - 0.25) + 894.8 - x[0]
    h4 = 1000*np.sin(x[2] - 0.25) + 1000*np.sin(x[2] - x[3] - 0.25) + 894.8 - x[1]
    h5 = 1000*np.sin(x[3] - 0.25) + 1000*np.sin(x[3] - x[2] - 0.25) + 1294.8

    return np.array([g1, g2, h3, h4, h5])

In [19]:
def g5(trial):
    x1 = trial.suggest_int("x1", 0, 1200)
    x2 = trial.suggest_int("x2", 0, 1200)
    x3 = trial.suggest_float("x3", -0.55, 0.55)
    x4 = trial.suggest_float("x4", -0.55, 0.55)
    x = np.array([x1, x2, x3, x4])
    cons = g5_cons(x)
    penalty_score = penalty(cons)
    obj = (3*x[0] + 0.000001*x[0]**3 + 2*x[1] + (0.000002/3)*(x[1]**3))
    return obj + penalty_score

def g5_og(x):
    x = np.array(x)
    obj = (3*x[0] + 0.000001*x[0]**3 + 2*x[1] + (0.000002/3)*(x[1]**3))
    return obj

In [20]:
def g6_cons(x):
    g1 = -(x[0] - 5)**2 - (x[1] - 5)**2 + 100
    g2 = 82.81 - (x[0] - 6)**2 - (x[1] - 5)**2

    return(np.array([g1,g2]))

In [21]:
def g6(trial):
    x1 = trial.suggest_int("x1", 0, 100)
    x2 = trial.suggest_int("x2", 13, 100)
    x = np.array([x1, x2])
    cons = g6_cons(x)
    penalty_score = penalty(cons)
    obj = ((x[0] - 10)**3 + (x[1] - 20)**3) 
    return obj + penalty_score

def g6_og(x):
    x = np.array(x)
    obj = ((x[0] - 10)**3 + (x[1] - 20)**3) 
    return obj 

In [22]:
def g7_cons(x):
    g1 = 4*x[0] + 5*x[1] - 3*x[6] + 9*x[7] - 105
    g2 = 10*x[0] - 8*x[1] - 17*x[6] + 2*x[7]
    g3 = -8*x[0] + 2*x[1] + 5*x[8] - 2*x[9] - 12
    g4 = 3*(x[0] - 2)**2 + 4*(x[1] - 3)**2 + 2*x[2]**2 - 7*x[3] - 120
    g5 = 5*x[0]**2 + 8*x[1] + (x[2] - 6)**2 - 2*x[3] - 40
    g6 = 0.5*(x[0] - 8)**2 + 2*(x[1] - 4)**2 + 3*x[4]**2 - x[5] - 30
    g7 = x[0]**2 + 2*(x[1] - 2)**2 - 2*x[0]*x[1] + 14*x[4] - 6*x[5]

    return np.array([g1, g2, g3, g4, g5, g6, g7])

In [23]:
def g7(trial):
    x1 = trial.suggest_float("x1", -10, 10)
    x2 = trial.suggest_float("x2", -10, 10)
    x3 = trial.suggest_float("x3", -10, 10)
    x4 = trial.suggest_float("x4", -10, 10)
    x5 = trial.suggest_int("x5", -10, 10)
    x6 = trial.suggest_int("x6", -10, 10)
    x7 = trial.suggest_int("x7", -10, 10)
    x8 = trial.suggest_int("x8", -10, 10)
    x9 = trial.suggest_float("x9", -10, 10)
    x10 = trial.suggest_float("x10", -10, 10)
    x = np.array([x1, x2, x3, x4, x5, x6, x7, x8, x9, x10])
    cons = g7_cons(x)
    penalty_score = penalty(cons)
    obj = (x[0]**2 + x[1]**2 + x[0]*x[1] - 14*x[0] - 16*x[1] + (x[2] - 10)**2 + 4*(x[3] - 5)**2 + (x[4] - 3)**2 + 2*(x[5] - 1)**2 + 5*x[6]**2 + 7*(x[7] - 11)**2 + 2*(x[8] - 10)**2 + (x[9] - 7)**2 + 45)
    return obj  + penalty_score

def g7_og(x):
    x = np.array(x)
    obj = (x[0]**2 + x[1]**2 + x[0]*x[1] - 14*x[0] - 16*x[1] + (x[2] - 10)**2 + 4*(x[3] - 5)**2 + (x[4] - 3)**2 + 2*(x[5] - 1)**2 + 5*x[6]**2 + 7*(x[7] - 11)**2 + 2*(x[8] - 10)**2 + (x[9] - 7)**2 + 45)
    return obj 


In [24]:
def g8_cons(x):
    g1 = x[0]**2 - x[1] + 1
    g2 = 1 - x[0] + (x[1] - 4)**2
    return np.array([g1, g2])

In [25]:
def g8(trial):
    x1 = trial.suggest_float("x1", 0, 10)
    x2 = trial.suggest_int("x2", 0, 10)
    x = np.array([x1, x2])
    cons = g8_cons(x)
    penalty_score = penalty(cons)
    obj = (np.sin(2 * np.pi * x[0])**3 * np.sin(2 * np.pi * x[1])) / (x[0]**3 * (x[0] + x[1])) 
    return -obj + penalty_score

def g8_og(x):
    x = np.array(x)
    obj = (np.sin(2 * np.pi * x[0])**3 * np.sin(2 * np.pi * x[1])) / (x[0]**3 * (x[0] + x[1])) 
    return -obj

In [26]:
def g9_cons(x):
    g1 = -127 + 2*x[0]**2 + 3*x[1]**4 + x[2] + 4*x[3]**2 + 5*x[4]
    g2 = -282 + 7*x[0] + 3*x[1] + 10*x[2]**2 + x[3] - x[4]
    g3 = -196 + 23*x[0] + x[1]**2 + 6*x[5]**2 - 8*x[6]
    g4 = 4*x[0]**2 + x[1]**2 - 3*x[0]*x[1] + 2*x[2]**2 + 5*x[5] - 11*x[6]

    return np.array([g1, g2, g3, g4])

In [27]:
def g9(trial):
    x1 = trial.suggest_float("x1", -10, 10)
    x2 = trial.suggest_float("x2", -10, 10)
    x3 = trial.suggest_float("x3", -10, 10)
    x4 = trial.suggest_float("x4", -10, 10)
    x5 = trial.suggest_float("x5", -10, 10)
    x6 = trial.suggest_int("x6", -10, 10)
    x7 = trial.suggest_int("x7", -10, 10)
    x = np.array([x1, x2, x3, x4, x5, x6, x7])
    cons = g9_cons(x)
    penalty_score = penalty(cons)
    obj = (x[0] - 10)**2 + 5*(x[1] - 12)**2 + x[2]**4 + 3*(x[3] - 11)**2 + 10*x[4]**6 + 7*x[5]**2 + x[6]**4 - 4*x[5]*x[6] - 10*x[5] - 8*x[6]
    return obj + penalty_score

def g9_og(x):
    x = np.array(x)
    obj = (x[0] - 10)**2 + 5*(x[1] - 12)**2 + x[2]**4 + 3*(x[3] - 11)**2 + 10*x[4]**6 + 7*x[5]**2 + x[6]**4 - 4*x[5]*x[6] - 10*x[5] - 8*x[6]
    return obj


In [28]:
def g10_cons(x):
    g1 = -1 + 0.0025*(x[3] + x[5])
    g2 = -1 + 0.0025*(x[4] + x[6] - x[3])
    g3 = -1 + 0.01*(x[7] - x[4])
    g4 = -x[0]*x[5] + 833.33252*x[3] + 100*x[0] - 83333.333
    g5 = -x[1]*x[6] + 1250*x[4] + x[1]*x[3] - 1250*x[3]
    g6 = -x[2]*x[7] + 1250000 + x[2]*x[4] - 2500*x[4]

    return np.sum([g1, g2, g3, g4, g5, g6])


In [29]:
def g10(trial):
    x1 = trial.suggest_float("x1", -10, 10)
    x2 = trial.suggest_float("x2", -10, 10)
    x3 = trial.suggest_float("x3", -10, 10)
    x4 = trial.suggest_float("x4", -10, 10)
    x5 = trial.suggest_float("x5", -10, 10)
    x6 = trial.suggest_float("x6", -10, 10)
    x7 = trial.suggest_float("x7", -10, 10)
    x8 = trial.suggest_float("x8", -10, 10)
    x = np.array([x1, x2, x3, x4, x5, x6, x7, x8])
    cons = g10_cons(x)
    penalty_score = penalty(cons)
    obj = x[0] + x[1] + x[2]
    return obj  + penalty_score

def g10_og(x):
    x = np.array(x)
    obj = x[0] + x[1] + x[2]
    return obj

In [30]:
def g11_cons(x):
    x = np.array(x)
    g1 = x[1] - x[0] ** 2

    return np.array([g1])


def g11(trial):
    x1 = trial.suggest_float("x1", -1, 1)
    x2 = trial.suggest_float("x2", -1, 1)
    x = np.array([x1, x2])
    cons = g11_cons(x)
    penalty_score = penalty(cons)
    obj = x[0] ** 2 + (x[1] - 2) ** 2
    return obj  +penalty_score

def g11_og(x):
    x = np.array(x)
    obj = x[0] ** 2 + (x[1] - 2) ** 2
    return obj 

In [31]:
n_trials = 2000
# functions = [g1, g2, g3, g4, g5, g6, g7, g8 ,g9, g10, g11]
functions = [g3]
random_seeds = [0, 2, 5, 50, 100]
relative_path = "results"
# fun_names = ["g1", "g2", "g3", "g4", "g5", "g6", "g7", "g8", "g9", "g10", "g11"]
fun_names = ["g3"]
# og_functions = [g1_og, g2_og, g3_og, g4_og, g5_og, g6_og, g7_og, g8_og, g9_og, g10_og, g11_og]
og_functions = [g3_og]
# cons_functions = [g1_cons, g2_cons, g3_cons, g4_cons, g5_cons, g6_cons, g7_cons, g8_cons, g9_cons, g10_cons, g11_cons]
cons_functions = [g3_cons]
sampler=CmaEsSampler(with_margin=True)

c:\Users\sanja\miniconda3\envs\bo\lib\site-packages\optuna\_experimental.py:30: ExperimentalWarning: Argument ``with_margin`` is an experimental feature. The interface can change in the future.
  warnings.warn(


In [32]:
for index, fn in enumerate(functions):
    og_fun = og_functions[index]
    fun_name = fun_names[index]
    cons_function = cons_functions[index]
    path = relative_path + "/" + fun_name
    isExist = os.path.exists(path)
    if not isExist:
        os.mkdir(path)
    for seed in random_seeds:
        fitness_scores = []
        best_individuals = []   
        sampler=CmaEsSampler(with_margin=True, seed = seed,            # Initial step size, as discussed
    restart_strategy='ipop', # Choose 'ipop' or 'bipop' depending on your needs
    popsize=13,              # Starting population size (you can tune this)
    inc_popsize=2 )
        study = optuna.create_study(sampler = sampler)
        res = study.optimize(fn, n_trials= n_trials)
        trials = study.get_trials()
        
        fopt_star = np.inf
        for trial_count ,trial in enumerate(trials):
            best_individual = list(trial.params.values())
            # print(f"trial {trial_count} {np.sum(cons_function(best_individual)) < 0}")
            viol = np.sum(cons_function(best_individual) > 0) 
            if (viol <= 0):
                fopt = og_fun(best_individual)
                if(fopt_star > fopt):
                    fopt_star = fopt.copy()
                # print("fopt star",{fopt_star})
                best_individuals.append(best_individual)
                fitness_scores.append(fopt_star)
        # print("fitness length ", seed)        
        np.savetxt(path + "/"+fun_name+"_"+str(seed)+".txt", fitness_scores)

c:\Users\sanja\miniconda3\envs\bo\lib\site-packages\optuna\_experimental.py:30: ExperimentalWarning: Argument ``restart_strategy`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2024-12-19 12:36:34,356] A new study created in memory with name: no-name-c9893453-f285-4ef8-907a-1cfc2025e1c6
[I 2024-12-19 12:36:34,359] Trial 0 finished with value: -10.209513477998879 and parameters: {'x1': 1, 'x2': 0.7151893663724195, 'x3': 0.6027633760716439, 'x4': 1, 'x5': 0.4236547993389047}. Best is trial 0 with value: -10.209513477998879.
[I 2024-12-19 12:36:35,127] Trial 1 finished with value: 0.0 and parameters: {'x1': 0, 'x2': 0.7914653467823916, 'x3': 0.508422112841139, 'x4': 0, 'x5': 0.04828293419129992}. Best is trial 0 with value: -10.209513477998879.
[I 2024-12-19 12:36:35,129] Trial 2 finished with value: 0.0 and parameters: {'x1': 0, 'x2': 0.27401439562197116, 'x3': 0.5555921003779606, 'x4': 1, 'x5': 0.7402452328087226}. Best is trial 0 with value: -1